# 🤖 TinyLLM-nano Kaggle トレーニング

**Kaggle GPU (T4 x2 / P100 16GB) で TinyLLM-nano (318M params) を追加事前学習します。**

## 特徴
- ✅ OOM 対策済み（batch=1, grad_accum=4, gradient checkpointing, シングルGPU推奨）
- ✅ HFからのチェックポイント自動ダウンロード＆再開
- ✅ HFへのチェックポイント自動アップロード
- ✅ 安全保存（`_use_new_zipfile_serialization=False`）
- ✅ 9時間制限を超えても継続学習可能

## 事前準備
1. Kaggle Notebook 設定: **Settings → Internet → ON**
2. Kaggle Secrets に `HF_TOKEN` を設定（HFアップロード用、オプション）
3. Accelerator: **T4 x2** または **P100**

## 🔧 Step 0: OOM対策 & 環境セットアップ

**重要**: CUDAメモリ断片化を防ぐため、`import torch` の前に設定します。

In [ ]:
# ============================================================
# Step 0: OOM対策 & インポート
# ============================================================
import os, sys, json, math, time, gc, argparse

# ★ CUDAメモリ断片化防止（torch import の前に！）
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from transformers import AutoTokenizer

print("=" * 60)
print("🔍 Kaggle GPU リソース診断")
print("=" * 60)

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"✅ GPU {i}: {props.name} ({props.total_memory/1e9:.1f} GB)")
    # ★ シングルGPU推奨（DataParallelはOOMの原因）
    print(f"\n💡 GPU {torch.cuda.device_count()}基検出。シングルGPUモードで実行します（OOM回避）")
    device = 'cuda'
    USE_BF16 = torch.cuda.get_device_capability()[0] >= 8
else:
    print("❌ GPUなし")
    device = 'cpu'
    USE_BF16 = False

import psutil
ram = psutil.virtual_memory()
print(f"📀 RAM: {ram.total/1e9:.1f} GB (free: {ram.available/1e9:.1f} GB)")

print(f"\n⚙️  PYTORCH_ALLOC_CONF={os.environ.get('PYTORCH_ALLOC_CONF', 'not set')}")
print(f"⚙️  BF16={'YES' if USE_BF16 else 'NO (FP16)'}")

## 🛡️ Step 0.5: 安全保存関数

Kaggle のファイルシステムでは `torch.save` の zip シリアライゼーションが破損することがあります。
レガシー形式 + テンポラリファイル経由のアトミック保存で回避します。

In [ ]:
def safe_save(obj, path, max_retries=3):
    """Kaggleで安全にチェックポイントを保存する。
    
    - _use_new_zipfile_serialization=False: レガシーpickle形式（Kaggleで安定）
    - .tmp に書き込んでから os.replace(): アトミック保存で破損防止
    - 最大3回リトライ
    """
    tmp_path = path + '.tmp'
    for i in range(max_retries):
        try:
            torch.save(obj, tmp_path, _use_new_zipfile_serialization=False)
            os.replace(tmp_path, path)
            return True
        except RuntimeError as e:
            print(f"⚠️  Save attempt {i+1}/{max_retries}: {e}")
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
            if i == max_retries - 1:
                raise
            time.sleep(2)
    return False

print("✅ safe_save() 定義完了")

## 📦 Step 1: パッケージインストール

In [ ]:
import subprocess, importlib

def check_and_install(pkg, pip_name=None):
    if pip_name is None:
        pip_name = pkg
    try:
        importlib.import_module(pkg)
        return True
    except ImportError:
        print(f"📥 {pip_name} インストール中...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        return False

for pkg in ['transformers', 'datasets', 'huggingface_hub', 'tqdm']:
    check_and_install(pkg)

print("✅ パッケージ準備完了")
print(f"   PyTorch: {torch.__version__}")
print(f"   Transformers: {__import__('transformers').__version__}")

## 🎯 Step 2: 設定

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 学習設定
# ═══════════════════════════════════════════════════════════════

HF_REPO = "Ryo3desu/tinyllm-models"
HF_MODEL_PATH = "tinyllm-nano/model.pt"
SEQ_LEN = 1024
OUTPUT_DIR = "checkpoints"

TRAIN_CONFIG = {
    'max_steps': 5000,        # 目標ステップ数
    'batch_size': 1,          # T4 OOM回避のため1に
    'grad_accum': 4,          # 実効バッチ = 1 × 4 = 4
    'learning_rate': 3e-4,
    'warmup_steps': 100,
    'max_lr': 3e-4,
    'min_lr': 3e-5,
    'weight_decay': 0.1,
    'grad_clip': 1.0,
    'log_interval': 10,
    'save_interval': 500,
    'gradient_checkpointing': True,
}

DATA_CONFIG = {
    'token_limit': 5_000_000,
}

print("📋 設定:")
for k, v in TRAIN_CONFIG.items():
    print(f"   {k}: {v}")
print(f"   実効バッチサイズ: {TRAIN_CONFIG['batch_size']} × {TRAIN_CONFIG['grad_accum']} = {TRAIN_CONFIG['batch_size'] * TRAIN_CONFIG['grad_accum']}")
print(f"   シーケンス長: {SEQ_LEN}")
print(f"   デバイス: {device.upper()}")

## 🏗️ Step 3: モデル定義

簡易 Transformer (RMSNorm + SwiGLU FFN + Scaled Dot-Product Attention)

In [ ]:
class TinyLLMLayer(nn.Module):
    """Single transformer block: RMSNorm → Attention → RMSNorm → SwiGLU FFN."""
    def __init__(self, cfg):
        super().__init__()
        D = cfg['hidden_size']
        inter = cfg.get('intermediate_size', D * 11 // 4)
        self.norm1 = nn.RMSNorm(D, eps=1e-5)
        self.norm2 = nn.RMSNorm(D, eps=1e-5)
        self.q_proj = nn.Linear(D, D, bias=False)
        self.k_proj = nn.Linear(D, D, bias=False)
        self.v_proj = nn.Linear(D, D, bias=False)
        self.o_proj = nn.Linear(D, D, bias=False)
        self.gate_proj = nn.Linear(D, inter, bias=False)
        self.up_proj   = nn.Linear(D, inter, bias=False)
        self.down_proj = nn.Linear(inter, D, bias=False)

    def forward(self, x):
        residual = x
        x = self.norm1(x)
        B, S, D = x.shape
        n_heads = 16
        head_dim = D // n_heads
        q = self.q_proj(x).view(B, S, n_heads, head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, S, n_heads, head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, n_heads, head_dim).transpose(1, 2)
        attn = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = self.o_proj(attn.transpose(1, 2).contiguous().view(B, S, D)) + residual
        residual = x
        x = self.norm2(x)
        x = self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)) + residual
        return x


class TinyLLMModel(nn.Module):
    """TinyLLM — 318M param simple transformer."""
    def __init__(self, cfg):
        super().__init__()
        self.embed = nn.Embedding(cfg['vocab_size'], cfg['hidden_size'])
        self.layers = nn.ModuleList([TinyLLMLayer(cfg) for _ in range(cfg['num_hidden_layers'])])
        self.norm = nn.RMSNorm(cfg['hidden_size'], eps=1e-5)
        self.lm_head = nn.Linear(cfg['hidden_size'], cfg['vocab_size'], bias=False)

    def forward(self, input_ids, labels=None):
        x = self.embed(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)
        return {'loss': loss, 'logits': logits}

print("✅ TinyLLMModel 定義完了")

## 📥 Step 4: トークナイザー & データ準備

In [ ]:
# ── トークナイザー ──
MODEL_DIR = 'downloaded_models/tinyllm-nano'
os.makedirs(MODEL_DIR, exist_ok=True)

# HFからトークナイザーをダウンロード（なければローカルを使用）
try:
    from huggingface_hub import hf_hub_download
    for fname in ['tokenizer.json', 'tokenizer_config.json', 'config.json']:
        if not os.path.exists(f'{MODEL_DIR}/{fname}'):
            hf_hub_download(HF_REPO, f'tinyllm-nano/{fname}', local_dir='downloaded_models')
    print("✅ HF から設定ファイルをダウンロード")
except Exception as e:
    print(f"⚠️  HF download failed: {e}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or '</s>'
VOCAB_SIZE = len(tokenizer)
print(f"✅ Tokenizer: vocab={VOCAB_SIZE}")

# ── データ準備 ──
os.makedirs('data', exist_ok=True)

class TokenBinDataset(IterableDataset):
    def __init__(self, path, seq_len, vocab_size):
        self.data = np.memmap(path, dtype=np.int32, mode='r')
        self.seq_len = seq_len
        self.vocab_size = vocab_size

    def __iter__(self):
        while True:
            offset = np.random.randint(0, len(self.data) - self.seq_len - 1)
            tokens = self.data[offset:offset + self.seq_len + 1]
            input_ids = torch.from_numpy(tokens[:self.seq_len].astype(np.int64))
            labels = torch.from_numpy(tokens[1:self.seq_len + 1].astype(np.int64))
            mask = (input_ids >= 0) & (input_ids < self.vocab_size)
            labels[~mask] = -100
            yield {'input_ids': input_ids, 'labels': labels}

if not os.path.exists('data/train.bin'):
    print("📦 データセットをダウンロード中...")
    from datasets import load_dataset
    try:
        ds = load_dataset("codeparrot/codeparrot-clean", split="train", streaming=True).take(10000)
    except Exception:
        ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train", streaming=True).take(10000)
    
    all_tokens = []
    for sample in ds:
        text = sample.get('content') or sample.get('text') or ''
        if len(text) < 10:
            continue
        all_tokens.extend(tokenizer.encode(text))
        if len(all_tokens) >= DATA_CONFIG['token_limit']:
            break
    
    tokens = np.array(all_tokens, dtype=np.int32)
    split = int(len(tokens) * 0.9)
    tokens[:split].tofile('data/train.bin')
    tokens[split:].tofile('data/val.bin')
    print(f"✅ データ準備完了: train={split:,}, val={len(tokens)-split:,} tokens")
else:
    data = np.memmap('data/train.bin', dtype=np.int32, mode='r')
    print(f"✅ 既存データ: {len(data):,} tokens")

## 🧠 Step 5: モデルロード（HFからチェックポイント自動再開）

In [ ]:
# ── 設定 ──
cfg_dict = {
    'hidden_size': 1024,
    'num_hidden_layers': 24,
    'vocab_size': VOCAB_SIZE,
    'intermediate_size': 2816,
}

model = TinyLLMModel(cfg_dict)
start_step = 0

# HF から最新チェックポイントをダウンロード
try:
    from huggingface_hub import hf_hub_download
    ckpt_path = hf_hub_download(HF_REPO, HF_MODEL_PATH, local_dir='downloaded_models')
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    
    if 'model_state_dict' in ckpt:
        state_dict = ckpt['model_state_dict']
    else:
        state_dict = ckpt
    
    # DDP/FSDP の 'module.' プレフィックスを除去
    state_dict = {k.removeprefix('module.'): v for k, v in state_dict.items()}
    
    model.load_state_dict(state_dict, strict=False)
    start_step = ckpt.get('step', 0)
    print(f"✅ HF チェックポイントから再開: step {start_step:,}")
except Exception as e:
    print(f"⚠️  HF checkpoint なし ({e})。スクラッチから開始します。")

total_params = sum(p.numel() for p in model.parameters())
print(f"📊 モデル: {total_params/1e6:.0f}M params, {cfg_dict['num_hidden_layers']} layers, hidden={cfg_dict['hidden_size']}")

model = model.to(device)

## ⚡ Step 6: Gradient Checkpointing 有効化（VRAM削減）

In [ ]:
if TRAIN_CONFIG.get('gradient_checkpointing', False):
    from torch.utils.checkpoint import checkpoint
    for layer in model.layers:
        original_forward = layer.forward
        layer._original_forward = original_forward
        layer.forward = lambda x, l=layer: checkpoint(l._original_forward, x, use_reentrant=False)
    print("🧠 Gradient checkpointing: ON（VRAM ~30%削減）")
else:
    print("🧠 Gradient checkpointing: OFF")

## 🏃 Step 7: トレーニングループ

In [ ]:
# ── Optimizer ──
try:
    optimizer = AdamW(model.parameters(), lr=TRAIN_CONFIG['learning_rate'],
                     weight_decay=TRAIN_CONFIG['weight_decay'],
                     betas=(0.9, 0.95), eps=1e-8, fused=True)
except (TypeError, RuntimeError):
    optimizer = AdamW(model.parameters(), lr=TRAIN_CONFIG['learning_rate'],
                     weight_decay=TRAIN_CONFIG['weight_decay'],
                     betas=(0.9, 0.95), eps=1e-8)

# ── Scheduler (Cosine) ──
def lr_lambda(step):
    warmup = TRAIN_CONFIG['warmup_steps']
    max_s = TRAIN_CONFIG['max_steps']
    if step < warmup:
        return float(step) / max(1, warmup)
    progress = float(step - warmup) / max(1, max_s - warmup)
    return TRAIN_CONFIG['min_lr'] / TRAIN_CONFIG['max_lr'] + \
           (1 - TRAIN_CONFIG['min_lr'] / TRAIN_CONFIG['max_lr']) * 0.5 * (1 + math.cos(math.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda)
for _ in range(start_step):
    scheduler.step()

scaler = torch.cuda.amp.GradScaler()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

# ── DataLoader ──
train_dataset = TokenBinDataset('data/train.bin', SEQ_LEN, VOCAB_SIZE)
train_loader = DataLoader(train_dataset, batch_size=TRAIN_CONFIG['batch_size'])

# ── Training ──
model.train()
data_iter = iter(train_loader)
global_step = start_step
total_loss = 0.0
start_time = time.time()

print("=" * 60)
print(f"🚀 Training: step {start_step:,} → {TRAIN_CONFIG['max_steps']:,}")
print(f"   Device: {device.upper()}, AMP: {'BF16' if USE_BF16 else 'FP16'}")
print(f"   Batch: {TRAIN_CONFIG['batch_size']}, Grad Accum: {TRAIN_CONFIG['grad_accum']}")
print(f"   Effective batch: {TRAIN_CONFIG['batch_size'] * TRAIN_CONFIG['grad_accum']}")
print("=" * 60)

while global_step < TRAIN_CONFIG['max_steps']:
    # ── Data ──
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        batch = next(data_iter)

    input_ids = batch['input_ids'].to(device)
    labels = batch['labels'].to(device)

    # ── Forward ──
    with torch.cuda.amp.autocast(dtype=AMP_DTYPE):
        outputs = model(input_ids=input_ids, labels=labels)
        loss = outputs['loss']
    
    loss = loss / TRAIN_CONFIG['grad_accum']
    
    # ── Backward ──
    if USE_BF16:
        loss.backward()
    else:
        scaler.scale(loss).backward()
    total_loss += loss.item()

    # ── Step ──
    if (global_step + 1) % TRAIN_CONFIG['grad_accum'] == 0:
        if USE_BF16:
            torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CONFIG['grad_clip'])
            optimizer.step()
        else:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CONFIG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()
        optimizer.zero_grad()

    global_step += 1

    # ── Log ──
    if global_step % TRAIN_CONFIG['log_interval'] == 0:
        avg_loss = total_loss / TRAIN_CONFIG['log_interval']
        elapsed = time.time() - start_time
        tok_per_sec = (global_step - start_step) * TRAIN_CONFIG['batch_size'] * SEQ_LEN / elapsed if elapsed > 0 else 0
        gpu_mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
        print(f"   step {global_step:>6,} | loss={avg_loss:.4f} | {tok_per_sec:>5.0f} tok/s | GPU={gpu_mem:.1f}GB | {elapsed:>5.0f}s")
        total_loss = 0.0

    # ── Save ──
    if global_step % TRAIN_CONFIG['save_interval'] == 0:
        ckpt_dir = f'{OUTPUT_DIR}/step_{global_step}'
        os.makedirs(ckpt_dir, exist_ok=True)
        safe_save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'config': cfg_dict,
            'step': global_step,
        }, f'{ckpt_dir}/model.pt')
        print(f"💾 Saved: {ckpt_dir}/")

elapsed = time.time() - start_time
print(f"\n✅ Training Complete! {elapsed:.0f}s total")
print(f"   Steps: {start_step:,} → {global_step:,}")

## 💾 Step 8: 最終モデル保存

In [ ]:
final_dir = f'{OUTPUT_DIR}/final'
os.makedirs(final_dir, exist_ok=True)

# モデル重み保存（safe_save 使用）
safe_save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': cfg_dict,
    'step': global_step,
}, f'{final_dir}/model.pt')

# トークナイザー & config 保存
tokenizer.save_pretrained(final_dir)
with open(f'{final_dir}/config.json', 'w') as f:
    json.dump(cfg_dict, f, indent=2)

print(f"💾 Final model: {final_dir}/ (step {global_step:,})")
print(f"   Files: {os.listdir(final_dir)}")

## ☁️ Step 9: HuggingFace にアップロード

Kaggle Secrets に `HF_TOKEN` を設定しておくと自動アップロードされます。

In [ ]:
# HF_TOKEN の取得
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    print("✅ Kaggle Secrets から HF_TOKEN を取得")
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

if not hf_token:
    print("⚠️  HF_TOKEN が設定されていません。アップロードをスキップします。")
    print("   設定方法: Kaggle → Add-ons → Secrets → HF_TOKEN を追加")
else:
    try:
        from huggingface_hub import login, HfApi
        login(token=hf_token)
        api = HfApi()
        
        size_gb = os.path.getsize(f'{final_dir}/model.pt') / 1e9
        print(f"📤 アップロード中 ({size_gb:.2f} GB)...")
        
        api.upload_file(
            path_or_fileobj=f'{final_dir}/model.pt',
            path_in_repo=HF_MODEL_PATH,
            repo_id=HF_REPO,
            repo_type="model",
        )
        print(f"✅ HF にアップロード完了: {HF_REPO}")
        print(f"   次回実行時に自動再開されます")
    except Exception as e:
        print(f"❌ アップロード失敗: {e}")

## 🧪 Step 10: 簡易推論テスト

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_tokens=128, temperature=0.7):
    model.eval()
    input_ids = tokenizer.encode(prompt)
    if len(input_ids) == 0:
        input_ids = [tokenizer.bos_token_id or 0]
    input_tensor = torch.tensor([input_ids], dtype=torch.long, device=device)
    
    for _ in range(max_tokens):
        if input_tensor.size(1) > SEQ_LEN:
            input_tensor = input_tensor[:, -SEQ_LEN:]
        logits = model(input_tensor)['logits'][0, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1).item()
        if next_token == tokenizer.eos_token_id:
            break
        input_tensor = torch.cat([input_tensor, torch.tensor([[next_token]], device=device)], dim=1)
    
    return tokenizer.decode(input_tensor[0].tolist(), skip_special_tokens=True)

print("🧪 推論テスト:")
prompt = "def fibonacci(n):"
print(f"   Prompt: {prompt}")
result = generate(model, tokenizer, prompt)
print(f"   Output: {result[:200]}")

## 📊 まとめ

- 学習済みモデルは `checkpoints/final/` に保存されています
- HF_TOKEN を設定していれば自動的に HF にアップロードされます
- 9時間制限で中断しても、次回実行時に自動再開します
- さらに学習を続けるには、`TRAIN_CONFIG['max_steps']` を増やして再実行してください